In [4]:
from transformers import ViTForImageClassification, ViTImageProcessor
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import random_split
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
from torch.utils.data import Dataset

import os
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [5]:
path = Path(os.path.join('..', 'input', 'sgfood-train-test', 'datasets'))
path_train = path/'train'
path_test = path/'test'

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model_name = "google/vit-base-patch16-224"
model = ViTForImageClassification.from_pretrained(model_name, image_size=160, num_labels=78, ignore_mismatched_sizes=True).to(device)
processor = ViTImageProcessor.from_pretrained(model_name)

new_dropout_rate = 0.2
def change_dropout(model, new_rate):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = new_rate

change_dropout(model, new_dropout_rate)

# Freeze ViT backbone
for param in model.vit.parameters():
    param.requires_grad = False

# Create a wrapper model with an additional layer and batch norm
class ViTWithExtraLayer(nn.Module):
    def __init__(self, base_model, hidden_classes, final_classes):
        super().__init__()
        self.base = base_model
        self.batch_norm = nn.BatchNorm1d(hidden_classes)  # Apply BatchNorm for logits
        self.extra_layer = nn.Linear(hidden_classes, final_classes)

    def forward(self, **kwargs):
        outputs = self.base(**kwargs)  # Get base model outputs
        logits_78 = outputs.logits  # Extract logits (78 classes from base model)

        # Apply BatchNorm on the logits
        logits_78 = self.batch_norm(logits_78)

        # Pass through the extra layer for final classification
        logits_4 = self.extra_layer(logits_78)

        return logits_4

# Wrap the model
model = ViTWithExtraLayer(model, hidden_classes=78, final_classes=4).to(device)

model = torch.nn.DataParallel(model)
model.to(device)

preprocess_train = Compose([
    Resize((160, 160)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

preprocess_test = Compose([
    Resize((160, 160)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

cuda


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([78]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([78, 768]) in the model instantiated
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 101, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_accuracy):
        if self.best_score is None or val_accuracy > self.best_score:
            self.best_score = val_accuracy
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [8]:
def evaluate_model(model, data_loader):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(pixel_values=inputs)

            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0.0)
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return running_loss / len(data_loader), accuracy * 100, precision, recall, f1

In [9]:
def train_model(model, train_loader, valid_loader, epochs=30):
    early_stopping = EarlyStopping(patience=3)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        total = 0
        correct = 0

        with tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", unit="batch") as tepoch:
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
    
                optimizer.zero_grad()
                outputs = model(pixel_values=inputs)
    
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
    
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total

        valid_loss, valid_accuracy, _, _, _ = evaluate_model(model, valid_loader)

        scheduler.step(valid_loss)

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {running_loss/len(train_loader):.4f}, Accuracy: {train_accuracy:.2f}%")
        print(f"Valid Loss: {valid_loss:.4f}, Accuracy: {valid_accuracy:.2f}%")
        print("-" * 40)

        early_stopping(valid_accuracy)
        if early_stopping.early_stop:
            print("Training stopped because of no increase in validation accuracy.")
            break

In [10]:
class NutriGradeDataset(Dataset):
    def __init__(self, dataset, category_to_nutri_grade, nutri_grade_to_numeric):
        self.dataset = dataset
        self.category_to_nutri_grade = category_to_nutri_grade
        self.nutri_grade_to_numeric = nutri_grade_to_numeric

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        class_name = self.dataset.classes[label]
        nutri_grade = self.category_to_nutri_grade[class_name]
        nutri_idx = self.nutri_grade_to_numeric[nutri_grade]

        return img, nutri_idx

In [11]:
category_to_nutri_grade = {
    'Apple': 'A',
    'Apricot': 'A',
    'banana': 'A',
    'Blackberry': 'A',
    'blueberries': 'A',
    'Papaya': 'A',
    'orange': 'A',
    'pear': 'A',
    'salad': 'A',
    'mixed vegetables': 'A',
    'green leafy vegetables': 'A',
    'sandwich': 'A',
    'salmon - grilled': 'A',
    'Soft boiled eggs': 'A',
    'milk': 'A',
    'Nuts': 'A',
    'whole grain bread': 'A',
    'whole oats': 'A',
    'cooked brown rice': 'A',
    'cooked white rice': 'A',
    'corn': 'A',
    'Porridge': 'A',
    'yogurt': 'A',
    'thunder tea rice': 'A',

    'steamed grouper': 'B',
    'Ban Mian': 'B',
    'bee hoon': 'B',
    'Udon': 'B',
    'Fish Ball Noodles': 'B',
    'Seafood Noodles Soup': 'B',
    'Prawn Noodle': 'B',
    'sirloin steak': 'B',
    'pasta - red sauce': 'B',
    'dumpling': 'B',
    'siew mai': 'B',
    'Bibimbap': 'B',
    'chicken soup': 'B',
    'muesli': 'B',
    'popiah': 'B',
    'kebab - chicken': 'B',
    'sushi': 'B',
    'roasted chicken': 'B',
    'otak': 'B',

    'Lor mee': 'C',
    'Mee rebus': 'C',
    'Mee siam': 'C',
    'nasi lemak': 'C',
    'bak kut teh': 'C',
    'Duck Rice': 'C',
    'Claypot Rice': 'C',
    'rice dumpling': 'C',
    'pineapple tarts': 'C',
    'Miso ramen, with fishcake': 'C',
    'chwee kueh': 'C',
    'chicken rice': 'C',
    'Hor Fun': 'C',
    'hokkien prawn mee': 'C',
    'goreng pisang': 'C',
    'tacos and nachos': 'C',

    'Burger': 'D',
    'sambal stingray': 'D',
    'oyster omelette': 'D',
    'cheese fries': 'D',
    'bak kwa': 'D',
    'chilli crab': 'D',
    'black pepper crab': 'D',
    'fish head curry': 'D',
    'Indian Prata': 'D',
    'ayam penyet': 'D',
    'Kway Teow': 'D',
    'Fish and chips': 'D',
    'fried chicken': 'D',
    'har cheong gai': 'D',
    'satay bee hoon': 'D',
    'ice kacang': 'D',
    'Laksa': 'D',
    'Chinese fritters': 'D',
    'curry puff': 'D'
}

nutri_grade_to_numeric = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3
}

In [12]:
data = ImageFolder(root=path_train, transform=preprocess_train)
data_converted = NutriGradeDataset(data, category_to_nutri_grade, nutri_grade_to_numeric)

val_size = int(0.2 * len(data_converted))
train_size = len(data_converted) - val_size
train_data, val_data = random_split(data_converted, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=150, shuffle=True, num_workers=2)
valid_loader = DataLoader(val_data, batch_size=150, shuffle=False, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
criterion = torch.nn.CrossEntropyLoss()

In [13]:
train_model(model, train_loader, valid_loader, epochs=30)

Epoch 1/30:   0%|          | 0/165 [01:45<?, ?batch/s]


Epoch 1/30
Train Loss: 1.0886, Accuracy: 52.88%
Valid Loss: 1.1052, Accuracy: 52.36%
----------------------------------------


Epoch 2/30:   0%|          | 0/165 [01:33<?, ?batch/s]


Epoch 2/30
Train Loss: 1.0228, Accuracy: 56.92%
Valid Loss: 1.0918, Accuracy: 53.50%
----------------------------------------


Epoch 3/30:   0%|          | 0/165 [01:32<?, ?batch/s]


...

Epoch 5/30
Train Loss: 1.0049, Accuracy: 57.68%
Valid Loss: 1.1034, Accuracy: 53.03%
----------------------------------------


Epoch 6/30:   0%|          | 0/165 [01:33<?, ?batch/s]


Epoch 6/30
Train Loss: 0.9939, Accuracy: 58.19%
Valid Loss: 1.1064, Accuracy: 53.25%
----------------------------------------


Epoch 7/30:   0%|          | 0/165 [01:32<?, ?batch/s]


Epoch 7/30
Train Loss: 1.0005, Accuracy: 58.03%
Valid Loss: 1.0917, Accuracy: 53.76%
----------------------------------------
Training stopped because of no increase in validation accuracy.


In [14]:
def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)
    print(f"Model saved to {filepath}")

save_model(model, '/kaggle/working/transformer_78_classes_to_4_classes_modified.pth')

Model saved to /kaggle/working/transformer_78_classes_to_4_classes_modified.pth


In [15]:
test_dataset = ImageFolder(path_test, transform=preprocess_test)
test_data_converted = NutriGradeDataset(data, category_to_nutri_grade, nutri_grade_to_numeric)
test_loader = DataLoader(test_data_converted, batch_size=150, shuffle=False)
loss, accuracy, precision, recall, f1 = evaluate_model(model, test_loader)

print(f"Test Accuracy: {accuracy/100:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

Test Accuracy: 0.5448
Test Precision: 0.5540
Test Recall: 0.5448
Test F1 Score: 0.5225
